<a href="https://colab.research.google.com/github/Ahmad-hub-bot/spendsense-ai/blob/main/notebooks/spendsense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai gspread google-auth

imports libs

In [2]:
import re
import requests
import pandas as pd
from datetime import datetime
from google import genai
import gspread
from google.oauth2.service_account import Credentials
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

Sample Training Data

In [3]:
# Balanced training set — 10 examples per category for classifier training
sample_transactions = [
    # FOOD (10)
    "INR 450.00 debited from your account for SWIGGY on 12-06-2026",
    "Rs 220 paid to ZOMATO for food order",
    "You spent Rs 320 at MCDONALDS via UPI",
    "Rs 150 paid to STARBUCKS via UPI",
    "INR 180 paid to DOMINOS PIZZA via UPI",
    "You spent Rs 75 at local TEA STALL via UPI",
    "Rs 500 spent at PIZZA HUT via UPI",
    "INR 90 debited at CAFE COFFEE DAY",
    "Rs 280 paid to KFC via UPI",
    "INR 60 debited at BAKERY for snacks",

    # SHOPPING (10)
    "Rs 1200 spent at ZARA using your card ending 4521",
    "INR 2500.00 debited at AMAZON for online purchase",
    "INR 3200.00 debited at H&M for shopping",
    "You spent Rs 1800 at DECATHLON",
    "Rs 950 spent at LIFESTYLE for clothing",
    "INR 1500 debited at MYNTRA for online order",
    "Rs 4200 spent at RELIANCE TRENDS",
    "INR 700 debited at NIKE STORE",
    "Rs 1100 paid to FLIPKART for purchase",
    "INR 2000 debited at PANTALOONS",

    # TRAVEL (10)
    "INR 89.50 debited for UBER ride on 13-06-2026",
    "INR 600 debited for OLA cab booking",
    "INR 99 debited for METRO CARD recharge",
    "INR 50 debited for PARKING fee",
    "Rs 4500 spent at FLIGHT BOOKING - INDIGO",
    "INR 6000.00 debited at MAKEMYTRIP for hotel booking",
    "Rs 60 paid for BUS TICKET via UPI",
    "INR 3000 debited for SPICEJET flight booking",
    "Rs 80 paid for AUTO RICKSHAW",
    "INR 250 debited for RAPIDO bike ride",

    # DAILY (10)
    "You spent Rs 95 at LOCAL GROCERY STORE",
    "Rs 40 paid for MILK delivery via UPI",
    "INR 120 debited at PHARMACY for medicines",
    "Rs 65 paid to VEGETABLE VENDOR via UPI",
    "INR 200 debited at BIG BAZAAR for groceries",
    "Rs 30 paid for NEWSPAPER subscription",
    "INR 150 debited at DMART for household items",
    "Rs 55 paid to LAUNDRY SERVICE",
    "INR 80 debited for WATER CAN delivery",
    "Rs 100 paid at GENERAL STORE",
]

labels = (
    ["food"] * 10 +
    ["shopping"] * 10 +
    ["travel"] * 10 +
    ["daily"] * 10
)

print(f"Total samples: {len(sample_transactions)}, Total labels: {len(labels)}")

Total samples: 40, Total labels: 40


Transaction Parser

In [4]:
def parse_transaction(sms_text):
    """Extracts amount and a rough merchant guess from raw SMS-style text."""
    amount_match = re.search(r'(?:INR|Rs\.?)\s?([\d,]+\.?\d*)', sms_text, re.IGNORECASE)
    amount = float(amount_match.group(1).replace(',', '')) if amount_match else None

    merchant_match = re.findall(r'\b[A-Z]{3,}(?:\s[A-Z]{2,})*\b', sms_text)
    ignore_words = {'INR', 'RS', 'UPI', 'CARD'}
    merchants = [m for m in merchant_match if m not in ignore_words]
    merchant = merchants[0] if merchants else "UNKNOWN"

    return {"amount": amount, "merchant": merchant, "raw_text": sms_text}

Category Classifier

In [5]:
# Stop words strip generic banking language so the model focuses on merchant names
vectorizer = TfidfVectorizer(stop_words=['inr', 'rs', 'debited', 'spent', 'paid', 'for', 'at', 'via',
                                            'upi', 'your', 'account', 'from', 'on', 'you', 'using', 'card', 'ending'])
X = vectorizer.fit_transform(sample_transactions)
y = labels

clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)  # trained on full dataset — too small to meaningfully split into train/test

def classify_transaction(sms_text):
    X_new = vectorizer.transform([sms_text])
    return clf.predict(X_new)[0]

# Quick sanity check on unseen merchants
test_examples = [
    "Rs 500 spent at SUBWAY via UPI",
    "INR 800 debited at LIFESTYLE for shopping",
    "Rs 300 paid for OLA AUTO ride",
    "INR 60 debited at LOCAL VEGETABLE SHOP",
]
for ex in test_examples:
    print(f"'{ex}' → {classify_transaction(ex)}")

'Rs 500 spent at SUBWAY via UPI' → food
'INR 800 debited at LIFESTYLE for shopping' → shopping
'Rs 300 paid for OLA AUTO ride' → travel
'INR 60 debited at LOCAL VEGETABLE SHOP' → daily


Budget Forecasting Logic

In [6]:
budgets = {
    "food": 1500,
    "shopping": 3000,
    "travel": 1000,
    "daily": 800
}

def forecast_breach(df, category, weekly_budget, week_start, week_end, today=None):
    """Checks current spend pace and forecasts if it'll exceed budget by week end."""
    if today is None:
        today = df['date'].max()

    week_data = df[(df['category'] == category) &
                    (df['date'] >= week_start) &
                    (df['date'] <= week_end)]

    current_spend = week_data['amount'].sum()
    days_elapsed = max((today.date() - week_start.date()).days + 1, 1)
    total_days = (week_end - week_start).days + 1

    daily_rate = current_spend / days_elapsed
    projected_total = daily_rate * total_days
    will_breach = projected_total > weekly_budget

    return {
        "category": category,
        "current_spend": round(current_spend, 2),
        "projected_total": round(projected_total, 2),
        "budget": weekly_budget,
        "will_breach": will_breach
    }

Telegram Alert Integration

In [7]:
bot_token = input("Paste your Telegram bot token: ")
chat_id = input("Paste your chat ID: ")

def send_telegram_alert(message):
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {"chat_id": chat_id, "text": message}
    return requests.post(url, data=payload).json()

Paste your Telegram bot token: 8923339991:AAEp4k7sjaF4I7isKdiVLFPksFe5fEERM08
Paste your chat ID: 8728889026


 Live Google Sheets Feed

In [9]:
json_filename = "/content/drive/MyDrive/Colab Notebooks/alpine-carrier-500101-m0-801b356c9b99.json"  # upload this file to Colab each session — never commit it to GitHub

scopes = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
creds = Credentials.from_service_account_file(json_filename, scopes=scopes)
client = gspread.authorize(creds)
sheet = client.open("SpendSense Live Feed").sheet1

 Full Pipeline — Sheets → Parse → Classify → Forecast → Alert


In [10]:
# Pull latest rows from the live feed
data = sheet.get_all_records()

processed_transactions = []
for row in data:
    sms_text = row['sms_text']
    parsed = parse_transaction(sms_text)
    category = classify_transaction(sms_text)
    processed_transactions.append({
        "date": row['timestamp'],
        "merchant": parsed['merchant'],
        "amount": parsed['amount'],
        "category": category
    })

live_df = pd.DataFrame(processed_transactions)
live_df['date'] = pd.to_datetime(live_df['date'])

# Anchor point for forecasting — uses latest available data instead of real-world now()
# to avoid date mismatch issues when running this notebook on a different day than the live data
simulated_today = live_df['date'].max()
week_start = live_df['date'].min().normalize()
week_end = week_start + pd.Timedelta(days=6)

# Note: with very few data points, projections can look aggressive (a single day's
# spend gets projected across the full week). This stabilizes as more days of data accumulate.
for cat, budget in budgets.items():
    result = forecast_breach(live_df, cat, budget, week_start, week_end, today=simulated_today)
    if result['will_breach']:
        alert_message = (
            f"⚠️ Budget Alert: {cat.upper()}\n"
            f"Spent so far: ₹{result['current_spend']}\n"
            f"Projected by week end: ₹{result['projected_total']}\n"
            f"Budget: ₹{result['budget']}\n"
            f"You're on pace to exceed this category's budget!"
        )
        send_telegram_alert(alert_message)
        print(f"🔔 Alert sent for {cat}")
    else:
        print(f"✅ {cat} on track — no alert needed")

🔔 Alert sent for food
🔔 Alert sent for shopping
✅ travel on track — no alert needed
✅ daily on track — no alert needed
